# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Rebuilds the Week-5 feature set and model so this notebook can audit it directly.

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "engagement_rate"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"{len(features)} features, {len(X):,} rows")

7 features, 30,000 rows


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: [*The State of AI-Driven SEO*, March 2026](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf) (341,701 content pieces, 57 brands; ML appendix on 61.8K active-content records).

---

### Finding: "What Predicts Health?" (ML Appendix, p.27)

**The claim:** a Random Forest predicting Health Score finds Average Position the top predictor (43% importance), then Impressions (32%), Scroll Depth (15%), and CTR (8%) — together 98% of the model's importance.

**Where does the label come from?** Health Score is explicitly defined earlier in the paper as `Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)` — a formula, not an observed outcome. And the top three features the model 'discovers' as most important (position, impressions, and CTR/scroll) are literally three of the label's four arithmetic components.

**Does the validation design carry the claim?** This is the constructive part, and the paper is already halfway there — it explicitly flags that "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal," and that high importance "does not imply external causation." That caveat is doing real work and is the right instinct. But a holdout split, by itself, doesn't fix this: a train/test split protects against overfitting to noise, not against a label that's a near-linear function of the features. A model can score perfectly on a clean holdout set and still just be re-deriving arithmetic, not finding anything about the world.

**Constructive suggestion:** re-run the same feature importance analysis using *only* the features that are NOT part of the Health Score formula — Content Age, Word Count, Days Visible, Sessions, AI Sessions (all already at 0% importance in the current chart, suspiciously). If those stay near zero even in a model that can't lean on the definitional overlap, that's a stronger, cleaner finding: "none of the non-definitional features predict health" is actually informative. Right now it's not possible to tell whether that 0% is a real null result or just those features losing a competition against three features that are mathematically guaranteed to win.

---

### Finding: "What Predicts Growth?" (ML Appendix, p.28)

**The claim:** a logistic regression reaches 71% holdout accuracy separating growing from declining pages, with Content Age as the strongest negative signal and Days Since Update / Days Visible as the strongest positive signals.

**Where does the label come from?** The Methodology section defines Trend Direction from 30-day-vs-previous-30-day impression change (Up >10%, Down >10%, Stable within ±10%) — the same kind of rule-based proxy label I've been using all along (my `is_declining_label` is built the same way, from `trend_direction`). No leakage risk here the way the Health Score finding has one — the listed features (age, days since update, days visible, position, word count, impressions, search volume, clicks, AI sessions, sessions) don't obviously overlap with how the label is computed.

**Does the validation design carry the claim?** This is the one I'd actually push on. The Methodology page states an "80/20 split" for the logistic regression, full stop — no mention of grouping by brand anywhere in the paper (I checked; the word 'grouped' never appears). The dataset spans 57 brands. If that 80/20 split is a plain random row split, the same brand's pages can land in both train and test, and the model could partly be learning "which brand is this" — its typical traffic tier, niche, update cadence — rather than a generalizable growth signal. I know this isn't hypothetical: Section 2 of this exact notebook reruns my own Week-5 model under a random split versus a client-grouped split on the same data, and precision@50 drops from 0.880 to 0.660 purely from the split change. A 71% accuracy on an ungrouped 61.8K-row, 57-brand split is very plausibly inflated the same way.

**Constructive suggestion:** re-run the logistic regression with a brand-grouped split (and report both numbers, the way I do below) — if 71% holds up under a grouped split, that's a genuinely stronger, more deployable claim. If it drops meaningfully, that's worth knowing before anyone uses "content age predicts growth" to make a brand-agnostic recommendation.

In [2]:
# No new query needed against my own data for this section — both methodology questions above
# are grounded in what the paper itself states (the Health Score formula, the 80/20 split language,
# and the absence of any grouped-split mention), plus the before/after evidence in Section 2 below.
print("See the markdown above for both findings and methodology questions.")

See the markdown above for both findings and methodology questions.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** a naive random row-level 75/25 split — the kind that looks standard if you don't think about it. **After:** the client-grouped split I actually used in Week 5. Both trained and scored the same way (Random Forest, same 7 features, same seed), so the only thing that changes between the two numbers is the split itself.

In [3]:
# BEFORE: naive random split — lets the same client appear in both train and test
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                    class_weight="balanced").fit(Xr_tr, yr_tr)
random_score = rf_random.predict_proba(Xr_te)[:, 1]

# AFTER: client-grouped split — same as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                     class_weight="balanced").fit(Xg_tr, yg_tr)
grouped_score = rf_grouped.predict_proba(Xg_te)[:, 1]

overlap = set(df["client_id"].iloc[Xr_tr.index]) & set(df["client_id"].iloc[Xr_te.index])
print(f"Clients in BOTH train and test under the random split: {len(overlap)} of "
      f"{df['client_id'].nunique()} total — this is exactly the memorization risk the honest split avoids.\n")

before_after = pd.DataFrame([
    {"split": "BEFORE — random row split",   "base_rate": round(yr_te.mean(), 3),
     "precision@20": round(precision_at_k(random_score, yr_te.values, 20), 3),
     "precision@50": round(precision_at_k(random_score, yr_te.values, 50), 3)},
    {"split": "AFTER — client-grouped split", "base_rate": round(yg_te.mean(), 3),
     "precision@20": round(precision_at_k(grouped_score, yg_te.values, 20), 3),
     "precision@50": round(precision_at_k(grouped_score, yg_te.values, 50), 3)},
])
before_after

Clients in BOTH train and test under the random split: 31 of 32 total — this is exactly the memorization risk the honest split avoids.



,split,base_rate,precision@20,precision@50
0,BEFORE — random row split,0.542,0.95,0.88
1,AFTER — client-grouped split,0.517,0.85,0.66


**Reading the gap:** precision@50 drops from 0.880 (random split) to 0.660 (grouped split) — a real, sizeable gap, not noise. That ~0.22 difference is a rough estimate of how much of the random-split score was the model partly recognizing *which client* a row belonged to (its typical traffic level, niche, position range) rather than genuinely reading decline. The grouped number is the one I'd defend to someone asking "will this work on a client you haven't seen yet" — which is the actual deployment question.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the attack checklist from the skill against my final 7-feature set:

- [x] **Timeline drawn:** all 7 features (`content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`, `engagement_rate`) are static or trailing-90-day aggregates, computed independently of the label window.
- [x] **No label-derived or sibling columns:** `trend_direction` and `trend_pct` are not in the feature list — verified programmatically below, not just by eye.
- [x] **No product flags:** the starter CSV has none to begin with (confirmed back in notebook 02).
- [x] **Grouped split:** used since Week 5, re-confirmed in Section 2 above.
- [x] **Base rate printed next to every metric:** done in every comparison table this notebook and the Week-5 one produce.
- [ ] → [x] **Deliberately add a leaky feature and watch the score jump:** this is the check below.

In [4]:
assert "trend_pct" not in features and "trend_direction" not in features

# Deliberately add trend_pct (the exact column the label is derived from) and re-score
X_leak = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
Xl_tr, Xl_te = X_leak.iloc[train_idx], X_leak.iloc[test_idx]

rf_leaky = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1,
                                   class_weight="balanced").fit(Xl_tr, yg_tr)
leaky_p50 = precision_at_k(rf_leaky.predict_proba(Xl_te)[:, 1], yg_te.values, 50)
honest_p50 = precision_at_k(grouped_score, yg_te.values, 50)

print(f"Honest P@50 (7 features):        {honest_p50:.3f}")
print(f"Leaky P@50 (+ trend_pct added):  {leaky_p50:.3f}  <- the confession")
print("\ntrend_pct is literally what trend_direction (my label) is bucketed from, so handing it")
print("to the model is handing it the answer. The jump to 1.000 confirms the test harness itself")
print("is working correctly — it WOULD catch leakage if it were present by accident.")

Honest P@50 (7 features):        0.660
Leaky P@50 (+ trend_pct added):  1.000  <- the confession

trend_pct is literally what trend_direction (my label) is bucketed from, so handing it
to the model is handing it the answer. The jump to 1.000 confirms the test harness itself
is working correctly — it WOULD catch leakage if it were present by accident.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, from Week 5:** *"Random forest wins clearly at @20; it ties logistic regression at @50 — both models beat the baseline and the dummy floor at both K, a real, reportable win."*

That sentence was already scoped to one held-out split and said so — but reading it again next to this notebook's before/after table, it doesn't carry the memorization-gap finding, and "wins clearly" reads more confident than an 8-client test set supports.

**Rewrite, in safe language:** *On a single client-grouped holdout (8 of 32 clients, n=7,115), the random forest showed a directional improvement over both the Week-4 rule baseline and a stratified-dummy floor at precision@20 and precision@50. This is an observed, decision-support result from one split, not a guarantee — a naive random split on the same data and model produced a notably higher, less trustworthy number (precision@50 0.880 vs. 0.660), which is itself a measured reminder that split choice can matter as much as model choice for this kind of claim.*

**What changed and why:** removed "wins clearly" (a confidence claim the sample size doesn't support), added the split size explicitly (n and client count, so a reader can judge precision themselves), and folded in the random-vs-grouped gap as context rather than leaving it as a separate, disconnected fact.

In [5]:
# No computation needed here — this section is a writing exercise applied to a claim
# already computed and verified earlier in this notebook and in w05_model.ipynb.
print("Claim rewritten above using only numbers already computed and shown in this notebook.")

Claim rewritten above using only numbers already computed and shown in this notebook.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.